# Consolidated Result Tables

This notebook builds four summary tables from `mrmr/results`:

1. Binary datasets (`openllm` + `helm`)
2. `continuous_cat_main` datasets
3. pass@k cross-k (target `k' <= 64`) with source `k=1`
4. pass@k cross-k (target `k' <= 64`) with source `k=opt` per method

Each table is exported to:

- parquet in `tables/`
- LaTeX in `tables/`

The LaTeX export uses horizontal rules only, groups rows by `(coreset_size, num_train_models)`, suppresses repeated group labels, and bolds best results (or values within one standard deviation of the best mean when std is available).

In [19]:
from __future__ import annotations

from pathlib import Path
import re
import sys
from typing import Iterable

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "mrmr").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing 'mrmr/'.")


PROJECT_ROOT = find_project_root(Path.cwd())
MRMR_DIR = PROJECT_ROOT / "mrmr"
DEFAULT_RESULTS_ROOT = MRMR_DIR / "results"
ALT_RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_ROOT = DEFAULT_RESULTS_ROOT if DEFAULT_RESULTS_ROOT.is_dir() else ALT_RESULTS_ROOT
TABLES_DIR = MRMR_DIR / "viz" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if str(MRMR_DIR) not in sys.path:
    sys.path.insert(0, str(MRMR_DIR))

from benchpred import all_methods  # noqa: E402
from data_utils import (  # noqa: E402
    openllm_datasets,
    helm_datasets,
    continuous_cat_main_datasets,
    pass_at_k_code_benchmarks,
    pass_at_k_v3_benchmarks,
)


# -----------------------------------------------------------------------------
# Experiment selection
# -----------------------------------------------------------------------------
BINARY_SPLIT_METHOD = "binned_interpolation"
CONTINUOUS_AND_PASSK_SPLIT_METHOD = "stratified"


# -----------------------------------------------------------------------------
# Method helpers + editable method list (one method per line)
# -----------------------------------------------------------------------------
def mrmr_family(
    *,
    prefixes: Iterable[str] = ("", "k", "k3", "k4"),
    mi_k_values: Iterable[int] = (4, 5, 6, 7, 8, 9),
    strategies: Iterable[str] = ("MID", "MIQ"),
    target: str = "y",
) -> list[str]:
    methods = []
    for prefix in prefixes:
        for mi_k in mi_k_values:
            for strategy in strategies:
                methods.append(f"{prefix}mrmr{mi_k}_{strategy}_{target}")
    return methods


METHODS_TO_SHOW = [
    # Baselines
    "random_sampling",
    "random_sampling_and_learn",
    "sample_first_and_learn",
    "random_search_and_learn",
    "small_search_and_learn",
    "aipw",
    "double_optimize",
    "lasso",
    "pca",
    "metabench",
    "anchor_points_weighted",
    "anchor_points_predictor",
    "pirt",
    "gpirt",
    "B_pirt",
    "B_gpirt",
    "B3_pirt",
    "B3_gpirt",
    "B3_v2_pirt",
    "B3_v2_gpirt",
    # Core MRMR variants
    "mrmr_MID_y",
    "mrmr_MIQ_y",
    "mrmr_MID_PC1",
    "mrmr_MIQ_PC1",
    "mrmr_MID_IRT1",
    "mrmr_MIQ_IRT1",
    "mrmr_FCD_y",
    "mrmr_FCQ_y",
    "mrmr_PMID_y",
    "mrmr_PMIQ_y",
    "mrmr_GMID_y",
    "mrmr_GMIQ_y",
    "mrmr_QGMID_y",
    "mrmr_QGMIQ_y",
    # Kernel / CV / RF / logit examples
    "kmrmr_MID_y",
    "kmrmr_MIQ_y",
    "k3mrmr_MID_y",
    "k3mrmr_MIQ_y",
    "k4mrmr_MID_y",
    "k4mrmr_MIQ_y",
    "cvmrmr_MID_y",
    "cvmrmr_MIQ_y",
    "rfmrmr_MID_y",
    "rfmrmr_MIQ_y",
    "lmrmr_MID_y",
    "lmrmr_MIQ_y",
]

# Optional compact family expansion (uncomment if useful):
# METHODS_TO_SHOW += mrmr_family(prefixes=("", "k", "k3", "k4"), mi_k_values=(4, 5, 6, 7, 8, 9), strategies=("MID", "MIQ"), target="y")

AVAILABLE_METHODS = set(all_methods.keys())
missing_methods = [m for m in METHODS_TO_SHOW if m not in AVAILABLE_METHODS]
METHODS_TO_SHOW = [m for m in METHODS_TO_SHOW if m in AVAILABLE_METHODS]

if missing_methods:
    print(f"Skipped {len(missing_methods)} unavailable methods.")


# -----------------------------------------------------------------------------
# Dataset lists (one entry per line)
# -----------------------------------------------------------------------------
BINARY_DATASETS = [
    # OpenLLM
    "ifeval",
    "openllm_math",
    "mmlu_pro",
    "arc_challenge",
    "bbh",
    "gpqa",
    "musr",
    # HELM
    "commonsense",
    "gsm",
    "legalbench",
    "math",
    "med_qa",
    "mmlu",
]

CONTINUOUS_MAIN_DATASETS = [
    "biolaysumm_rougel",
    "biolaysumm_bertscore",
    "biolaysumm_fkgl",
    "govreport_rougel",
    "govreport_bertscore",
    "truthfulqa_judge",
    "nemotron_pii",
]

PASSK_BENCHMARKS = [
    "MBPP_mbpp",
    "MBPPPlus_mbpp_plus",
    ,
    "HumanEvalPack_Rust",
    "HumanEvalPack_Java",
    "HumanEvalPack_Js",
    "HumanEvalPack_Go",
    "HumanEvalPack_Cpp",
    "HumanEvalPack_PythonPlus",
    "LBPP_Cpp",
    "LBPP_Java",
    "LBPP_Js",
    "LBPP_Go",
    "LBPP_Python",
    "LBPP_Rust",
]

PASSK_TARGET_KS = [
    1,
    2,
    4,
    8,
    16,
    32,
    64,
]
PASSK_ALLOWED_SUFFIXES = {"_v3", "_v3_open"}
PASSK_EXCLUDED_KS = {128}

# Used only for the source_k=opt table. Change if you want a different criterion.
PASSK_OPTIMIZATION_METRIC = "RMSE"  # one of: RMSE, MAE, Kendall_tau, Spearman_rho, pearson_r, execution_time


# -----------------------------------------------------------------------------
# Sanity checks against canonical lists
# -----------------------------------------------------------------------------
print(f"Project root: {PROJECT_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Tables dir:   {TABLES_DIR}")
print(f"Binary split method: {BINARY_SPLIT_METHOD}")
print(f"Continuous/pass@k split method: {CONTINUOUS_AND_PASSK_SPLIT_METHOD}")
print(f"Using {len(METHODS_TO_SHOW)} methods")

print("\nBinary datasets missing from canonical openllm+helm:")
print(sorted(set(BINARY_DATASETS) - set(openllm_datasets + helm_datasets)))

print("\nContinuous-main datasets missing from canonical list:")
print(sorted(set(CONTINUOUS_MAIN_DATASETS) - set(continuous_cat_main_datasets)))

all_passk_benchmarks = sorted(set(pass_at_k_v3_benchmarks))
print("\nPass@k benchmarks missing from canonical _v3 list:")
print(sorted(set(PASSK_BENCHMARKS) - set(all_passk_benchmarks)))

Project root: /home/dg22309/Documents/mrmr_project
Results root: /home/dg22309/Documents/mrmr_project/mrmr/results
Tables dir:   /home/dg22309/Documents/mrmr_project/mrmr/viz/tables
Binary split method: binned_interpolation
Continuous/pass@k split method: stratified
Using 46 methods

Binary datasets missing from canonical openllm+helm:
[]

Continuous-main datasets missing from canonical list:
[]

Pass@k benchmarks missing from canonical _v3 list:
['LBPP_Cpp', 'LBPP_Go', 'LBPP_Java', 'LBPP_Js', 'LBPP_Rust', 'HumanEvalPack_Cpp', 'HumanEvalPack_Go', 'HumanEvalPack_Java', 'HumanEvalPack_Js', 'HumanEvalPack_Rust']


In [20]:
# -----------------------------------------------------------------------------
# Loaders and aggregation helpers
# -----------------------------------------------------------------------------
_CONSOLIDATED_RESULTS = "_consolidated_results.parquet"
_CONSOLIDATED_CROSSK = "_consolidated_crossk.parquet"

METRICS = [
    ("RMSE", "rmse", "min"),
    ("MAE", "error", "min"),
    ("Kendall_tau", "corr_kendall", "max"),
    ("Spearman_rho", "corr_spearman", "max"),
    ("pearson_r", "corr_pearson", "max"),
    ("execution_time", "execution_time", "min"),
]

MEAN_COLS = [name for name, _, _ in METRICS]
SE_COLS = [f"{name}_se" for name, _, _ in METRICS]

PASSK_RE = re.compile(
    r"^(" + "|".join(re.escape(b) for b in sorted(set(PASSK_BENCHMARKS))) + r")_pass_at_(\d+)(_v3_open|_v3)$"
)


def parse_pass_at_k_dataset_name(dataset_name: str):
    m = PASSK_RE.match(dataset_name)
    if m is None:
        return None

    suffix = m.group(3)
    k = int(m.group(2))
    if suffix not in PASSK_ALLOWED_SUFFIXES or k in PASSK_EXCLUDED_KS:
        return None

    return m.group(1), k, suffix

def coreset_sort_key(value: str):
    s = str(value)
    if s.endswith("%"):
        return (1, float(s[:-1]))
    return (0, float(s))


def nmodels_sort_key(value: str):
    s = str(value)
    if s == "default":
        return (-1, -1)
    try:
        return (0, float(s))
    except ValueError:
        return (1, s)


def discover_settings(results_root: Path, split_methods: list[str] | None = None) -> list[dict]:
    settings = []
    if not results_root.is_dir():
        return settings

    for split_dir in sorted(p for p in results_root.iterdir() if p.is_dir()):
        split_method = split_dir.name
        if split_methods and split_method not in split_methods:
            continue

        for cs_dir in sorted(p for p in split_dir.iterdir() if p.is_dir() and p.name.startswith("coreset_")):
            coreset_size = cs_dir.name.replace("coreset_", "", 1)

            for nm_dir in sorted(p for p in cs_dir.iterdir() if p.is_dir() and p.name.startswith("nmodels_")):
                num_train_models = nm_dir.name.replace("nmodels_", "", 1)
                settings.append(
                    {
                        "split_method": split_method,
                        "coreset_size": coreset_size,
                        "num_train_models": num_train_models,
                        "nmodels_dir": nm_dir,
                    }
                )

    settings.sort(
        key=lambda x: (
            x["split_method"],
            coreset_sort_key(x["coreset_size"]),
            nmodels_sort_key(x["num_train_models"]),
        )
    )
    return settings


def load_setting_frames(nmodels_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    results_path = nmodels_dir / _CONSOLIDATED_RESULTS
    crossk_path = nmodels_dir / _CONSOLIDATED_CROSSK

    native_df = pd.read_parquet(results_path) if results_path.is_file() else pd.DataFrame()
    crossk_df = pd.read_parquet(crossk_path) if crossk_path.is_file() else pd.DataFrame()

    if not native_df.empty:
        native_df = native_df.copy()
        native_df["training_time"] = pd.to_numeric(native_df.get("training_time"), errors="coerce")
        native_df["inference_time"] = pd.to_numeric(native_df.get("inference_time"), errors="coerce")
        native_df["execution_time"] = native_df["training_time"].fillna(0.0) + native_df["inference_time"].fillna(0.0)

    return native_df, crossk_df


def summarize_from_unit_table(unit_df: pd.DataFrame, unit_col: str = "dataset") -> pd.DataFrame:
    if unit_df.empty:
        return pd.DataFrame(columns=["method", *MEAN_COLS, *SE_COLS])

    rows = {"method": sorted(unit_df["method"].unique())}
    summary = pd.DataFrame(rows)

    for metric_name, src_col, _ in METRICS:
        if src_col not in unit_df.columns:
            summary[metric_name] = np.nan
            summary[f"{metric_name}_se"] = np.nan
            continue

        stats = (
            unit_df.groupby("method", as_index=False)[src_col]
            .agg(["mean", "std", "count"])
            .reset_index()
            .rename(columns={"mean": metric_name})
        )
        stats[f"{metric_name}_se"] = stats["std"] / np.sqrt(stats["count"])
        stats = stats[["method", metric_name, f"{metric_name}_se"]]
        summary = summary.merge(stats, on="method", how="left")

    return summary


def summarize_standard_table(
    native_df: pd.DataFrame,
    datasets: list[str],
    methods: list[str],
) -> pd.DataFrame:
    if native_df.empty:
        return pd.DataFrame()

    sub = native_df[
        native_df["dataset"].isin(datasets)
        & native_df["method"].isin(methods)
    ].copy()
    if sub.empty:
        return pd.DataFrame()

    src_cols = [src for _, src, _ in METRICS if src in sub.columns]
    unit_df = (
        sub.groupby(["dataset", "method"], as_index=False)[src_cols]
        .mean()
    )
    return summarize_from_unit_table(unit_df, unit_col="dataset")


def build_passk_combined_frame(native_df: pd.DataFrame, crossk_df: pd.DataFrame) -> pd.DataFrame:
    if native_df.empty and crossk_df.empty:
        return pd.DataFrame()

    native_pass = pd.DataFrame()
    if not native_df.empty:
        native_pass = native_df.copy()
        native_pass["_parsed"] = native_pass["dataset"].map(parse_pass_at_k_dataset_name)
        native_pass = native_pass[native_pass["_parsed"].notna()].copy()
        if not native_pass.empty:
            native_pass["benchmark"] = native_pass["_parsed"].str[0]
            native_pass["source_k"] = native_pass["_parsed"].str[1]
            native_pass["pred_k"] = native_pass["source_k"]

    cross_aug = pd.DataFrame()
    if not crossk_df.empty:
        cross_aug = crossk_df.copy()

        if "benchmark" not in cross_aug.columns or "source_k" not in cross_aug.columns:
            cross_aug["_parsed"] = cross_aug["dataset"].map(parse_pass_at_k_dataset_name)
            cross_aug = cross_aug[cross_aug["_parsed"].notna()].copy()
            if not cross_aug.empty:
                cross_aug["benchmark"] = cross_aug["_parsed"].str[0]
                cross_aug["source_k"] = cross_aug["_parsed"].str[1]

        time_lookup = pd.DataFrame()
        if not native_pass.empty:
            time_lookup = native_pass[["dataset", "method", "seed", "execution_time"]].drop_duplicates()

        if not time_lookup.empty:
            cross_aug = cross_aug.merge(
                time_lookup,
                on=["dataset", "method", "seed"],
                how="left",
            )
        else:
            cross_aug["execution_time"] = np.nan

    common_cols = [
        "dataset",
        "benchmark",
        "method",
        "seed",
        "source_k",
        "pred_k",
        "error",
        "rmse",
        "corr_spearman",
        "corr_kendall",
        "corr_pearson",
        "execution_time",
    ]

    parts = []
    if not cross_aug.empty:
        parts.append(cross_aug.reindex(columns=common_cols))
    if not native_pass.empty:
        parts.append(native_pass.reindex(columns=common_cols))

    if not parts:
        return pd.DataFrame(columns=common_cols)

    combined = pd.concat(parts, ignore_index=True)
    combined = combined.dropna(subset=["benchmark", "source_k", "pred_k"], how="any")
    combined["source_k"] = pd.to_numeric(combined["source_k"], errors="coerce")
    combined["pred_k"] = pd.to_numeric(combined["pred_k"], errors="coerce")
    combined = combined.dropna(subset=["source_k", "pred_k"]) 
    combined["source_k"] = combined["source_k"].astype(int)
    combined["pred_k"] = combined["pred_k"].astype(int)
    combined = combined[
        ~combined["source_k"].isin(PASSK_EXCLUDED_KS)
        & ~combined["pred_k"].isin(PASSK_EXCLUDED_KS)
    ].copy()
    return combined


def summarize_passk_table(
    combined_df: pd.DataFrame,
    methods: list[str],
    benchmarks: list[str],
    target_ks: list[int],
    source_mode: str,
    opt_metric: str,
) -> pd.DataFrame:
    if combined_df.empty:
        return pd.DataFrame()

    sub = combined_df[
        combined_df["method"].isin(methods)
        & combined_df["benchmark"].isin(benchmarks)
        & combined_df["pred_k"].isin(target_ks)
    ].copy()
    if sub.empty:
        return pd.DataFrame()

    src_cols = [src for _, src, _ in METRICS if src in sub.columns]

    seed_avg = (
        sub.groupby(["benchmark", "method", "source_k", "pred_k"], as_index=False)[src_cols]
        .mean()
    )

    target_avg = (
        seed_avg.groupby(["benchmark", "method", "source_k"], as_index=False)[src_cols]
        .mean()
    )

    if source_mode == "k1":
        selected = target_avg[target_avg["source_k"] == 1].copy()
    elif source_mode == "opt":
        metric_info = {name: (src, direction) for name, src, direction in METRICS}
        if opt_metric not in metric_info:
            raise ValueError(f"Unknown opt metric: {opt_metric}")
        metric_src, direction = metric_info[opt_metric]

        method_source = (
            target_avg.groupby(["method", "source_k"], as_index=False)[metric_src]
            .mean()
        )

        if direction == "min":
            best_idx = method_source.groupby("method")[metric_src].idxmin()
        else:
            best_idx = method_source.groupby("method")[metric_src].idxmax()

        best_source = method_source.loc[best_idx, ["method", "source_k"]].rename(
            columns={"source_k": "selected_source_k"}
        )

        selected = target_avg.merge(
            best_source,
            left_on=["method", "source_k"],
            right_on=["method", "selected_source_k"],
            how="inner",
        )
    else:
        raise ValueError(f"Unknown source_mode: {source_mode}")

    if selected.empty:
        return pd.DataFrame()

    return summarize_from_unit_table(selected.rename(columns={"benchmark": "dataset"}), unit_col="dataset")


def apply_setting_columns(df: pd.DataFrame, setting: dict) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out.insert(0, "num_train_models", setting["num_train_models"])
    out.insert(0, "coreset_size", setting["coreset_size"])
    return out


def sort_final_table(df: pd.DataFrame, method_order: list[str]) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()
    out["_method_order"] = out["method"].map({m: i for i, m in enumerate(method_order)}).fillna(10**9)
    out["_cs_sort"] = out["coreset_size"].map(coreset_sort_key)
    out["_nm_sort"] = out["num_train_models"].map(nmodels_sort_key)

    out = out.sort_values(["_cs_sort", "_nm_sort", "_method_order", "method"])
    out = out.drop(columns=["_method_order", "_cs_sort", "_nm_sort"])

    keep_cols = ["coreset_size", "num_train_models", "method", *MEAN_COLS, *SE_COLS]
    return out[keep_cols].reset_index(drop=True)


_LATEX_ESCAPES = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def latex_escape(text: str) -> str:
    s = str(text)
    for old, new in _LATEX_ESCAPES.items():
        s = s.replace(old, new)
    return s


def best_and_within_se_masks(
    mean_vals: pd.Series,
    se_vals: pd.Series,
    direction: str,
) -> tuple[pd.Series, pd.Series]:
    mean_numeric = pd.to_numeric(mean_vals, errors="coerce")
    se_numeric = pd.to_numeric(se_vals, errors="coerce")

    valid = mean_numeric.notna()
    if not valid.any():
        empty_mask = pd.Series(False, index=mean_vals.index)
        return empty_mask, empty_mask

    best = mean_numeric[valid].min() if direction == "min" else mean_numeric[valid].max()
    best_mask = valid & np.isclose(mean_numeric, best, rtol=1e-12, atol=1e-12)

    tol = se_numeric.fillna(0.0)
    if direction == "min":
        within_mask = valid & ~best_mask & (mean_numeric <= best + tol)
    else:
        within_mask = valid & ~best_mask & (mean_numeric >= best - tol)

    return best_mask, within_mask


def format_uncertainty(value: float, metric_name: str) -> str | None:
    if pd.isna(value):
        return None
    if metric_name == "execution_time":
        return f"{value:.1e}"
    return f"{value:.2g}"


def format_metric_cell(mean_val: float, se_val: float, style: str, metric_name: str) -> str:
    if pd.isna(mean_val):
        return "--"

    if metric_name == "execution_time":
        mean_text = f"{mean_val:.2e}"
    else:
        mean_text = f"{mean_val:.4f}"

    if style == "bold":
        mean_text = f"\\textbf{{{mean_text}}}"
    elif style == "underline":
        mean_text = f"\\underline{{{mean_text}}}"

    se_text = format_uncertainty(se_val, metric_name)
    if se_text is not None:
        return f"{mean_text} {{\\scriptsize $\\pm$ {se_text}}}"
    return mean_text


def render_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    if df.empty:
        return "% Empty table\n"

    table_df = df.copy()

    # Highlight per (coreset_size, num_train_models) block:
    # - best mean: bold
    # - worse but within one SE of best: underline
    for (_, _), block_idx in table_df.groupby(["coreset_size", "num_train_models"], sort=False).groups.items():
        block_idx = list(block_idx)
        for metric_name, _, direction in METRICS:
            se_col = f"{metric_name}_se"
            best_mask, within_mask = best_and_within_se_masks(
                table_df.loc[block_idx, metric_name],
                table_df.loc[block_idx, se_col],
                direction,
            )
            for idx in block_idx:
                style = "normal"
                if bool(best_mask.loc[idx]):
                    style = "bold"
                elif bool(within_mask.loc[idx]):
                    style = "underline"

                table_df.loc[idx, metric_name] = format_metric_cell(
                    table_df.loc[idx, metric_name],
                    table_df.loc[idx, se_col],
                    style,
                    metric_name,
                )

    # Suppress repeated group labels for visual blocks.
    group_keys = ["coreset_size", "num_train_models"]
    grouped_indices = []
    for _, idxs in table_df.groupby(group_keys, sort=False).groups.items():
        idxs = list(idxs)
        grouped_indices.append(idxs)
        for idx in idxs[1:]:
            table_df.loc[idx, "coreset_size"] = ""
            table_df.loc[idx, "num_train_models"] = ""

    out_cols = ["coreset_size", "num_train_models", "method", *MEAN_COLS]
    header_map = {
        "coreset_size": "$n$",
        "num_train_models": "$M$",
        "method": "Method",
        "RMSE": "RMSE",
        "MAE": "MAE",
        "Kendall_tau": "Kendall $\\tau$",
        "Spearman_rho": "Spearman $\\rho$",
        "pearson_r": "Pearson $r$",
        "execution_time": "Execution Time",
    }
    header_row = " & ".join(header_map[c] for c in out_cols) + r" \\"

    lines = []
    lines.append(r"\footnotesize")
    lines.append(r"\setlength{\tabcolsep}{2pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.05}")
    lines.append(r"\setlength{\LTpre}{0pt}")
    lines.append(r"\setlength{\LTpost}{0pt}")
    # Keep the table centered after tightening column widths.
    lines.append(r"\setlength{\LTleft}{\fill}")
    lines.append(r"\setlength{\LTright}{\fill}")
    lines.append(r"\begin{longtable}{llp{0.15\textwidth}" + "r" * len(MEAN_COLS) + "}")
    lines.append(rf"\caption{{{latex_escape(caption)}}}\label{{{latex_escape(label)}}}\\")
    lines.append(r"\toprule")
    lines.append(header_row)
    lines.append(r"\midrule")
    lines.append(r"\endfirsthead")
    lines.append(r"\toprule")
    lines.append(header_row)
    lines.append(r"\midrule")
    lines.append(r"\endhead")
    lines.append(r"\midrule")
    lines.append(rf"\multicolumn{{{len(out_cols)}}}{{r}}{{\emph{{Continued on next page}}}} \\")
    lines.append(r"\midrule")
    lines.append(r"\endfoot")
    lines.append(r"\bottomrule")
    lines.append(r"\endlastfoot")

    block_start_indices = {idxs[0] for idxs in grouped_indices if idxs}
    first_row = True
    for idx, row in table_df[out_cols].iterrows():
        if idx in block_start_indices and not first_row:
            lines.append(r"\midrule")
        first_row = False

        cell_text = []
        for col in out_cols:
            value = row[col]
            if col in MEAN_COLS:
                cell_text.append(str(value))
            else:
                escaped = latex_escape(value)
                if col == "method":
                    escaped = escaped.replace(r"\_", r"\_\allowbreak")
                cell_text.append(escaped)
        lines.append(" & ".join(cell_text) + r" \\")

    lines.append(r"\end{longtable}")
    return "\n".join(lines)


def save_table(df: pd.DataFrame, stem: str, caption: str) -> tuple[Path, Path]:
    parquet_path = TABLES_DIR / f"{stem}.parquet"
    tex_path = TABLES_DIR / f"{stem}.tex"

    df.to_parquet(parquet_path, index=False)
    tex_str = render_latex_table(df, caption=caption, label=f"tab:{stem}")
    tex_path.write_text(tex_str)

    return parquet_path, tex_path

In [21]:
# -----------------------------------------------------------------------------
# Build all four tables and export artifacts
# -----------------------------------------------------------------------------
required_splits = sorted({BINARY_SPLIT_METHOD, CONTINUOUS_AND_PASSK_SPLIT_METHOD})
settings = discover_settings(RESULTS_ROOT, split_methods=required_splits)

if not settings:
    raise RuntimeError(f"No experiment settings found under {RESULTS_ROOT}")

print(f"Discovered {len(settings)} experiment settings across splits: {required_splits}")
print("Settings per split:")
for split_name in required_splits:
    n_split = sum(1 for s in settings if s["split_method"] == split_name)
    print(f"- {split_name}: {n_split}")

binary_rows = []
continuous_rows = []
passk_k1_rows = []
passk_opt_rows = []

for setting in settings:
    native_df, crossk_df = load_setting_frames(setting["nmodels_dir"])

    if native_df.empty and crossk_df.empty:
        continue

    if setting["split_method"] == BINARY_SPLIT_METHOD:
        # 1) Binary table (openllm + helm)
        binary_summary = summarize_standard_table(
            native_df=native_df,
            datasets=BINARY_DATASETS,
            methods=METHODS_TO_SHOW,
        )
        binary_summary = apply_setting_columns(binary_summary, setting)
        if not binary_summary.empty:
            binary_rows.append(binary_summary)

    if setting["split_method"] == CONTINUOUS_AND_PASSK_SPLIT_METHOD:
        # 2) Continuous-main table
        continuous_summary = summarize_standard_table(
            native_df=native_df,
            datasets=CONTINUOUS_MAIN_DATASETS,
            methods=METHODS_TO_SHOW,
        )
        continuous_summary = apply_setting_columns(continuous_summary, setting)
        if not continuous_summary.empty:
            continuous_rows.append(continuous_summary)

        # 3) and 4) pass@k cross-k tables
        passk_combined = build_passk_combined_frame(native_df=native_df, crossk_df=crossk_df)

        passk_k1_summary = summarize_passk_table(
            combined_df=passk_combined,
            methods=METHODS_TO_SHOW,
            benchmarks=PASSK_BENCHMARKS,
            target_ks=PASSK_TARGET_KS,
            source_mode="k1",
            opt_metric=PASSK_OPTIMIZATION_METRIC,
        )
        passk_k1_summary = apply_setting_columns(passk_k1_summary, setting)
        if not passk_k1_summary.empty:
            passk_k1_rows.append(passk_k1_summary)

        passk_opt_summary = summarize_passk_table(
            combined_df=passk_combined,
            methods=METHODS_TO_SHOW,
            benchmarks=PASSK_BENCHMARKS,
            target_ks=PASSK_TARGET_KS,
            source_mode="opt",
            opt_metric=PASSK_OPTIMIZATION_METRIC,
        )
        passk_opt_summary = apply_setting_columns(passk_opt_summary, setting)
        if not passk_opt_summary.empty:
            passk_opt_rows.append(passk_opt_summary)


binary_table_df = sort_final_table(
    pd.concat(binary_rows, ignore_index=True) if binary_rows else pd.DataFrame(columns=["coreset_size", "num_train_models", "method", *MEAN_COLS, *SE_COLS]),
    METHODS_TO_SHOW,
)

continuous_table_df = sort_final_table(
    pd.concat(continuous_rows, ignore_index=True) if continuous_rows else pd.DataFrame(columns=["coreset_size", "num_train_models", "method", *MEAN_COLS, *SE_COLS]),
    METHODS_TO_SHOW,
)

passk_k1_table_df = sort_final_table(
    pd.concat(passk_k1_rows, ignore_index=True) if passk_k1_rows else pd.DataFrame(columns=["coreset_size", "num_train_models", "method", *MEAN_COLS, *SE_COLS]),
    METHODS_TO_SHOW,
)

passk_opt_table_df = sort_final_table(
    pd.concat(passk_opt_rows, ignore_index=True) if passk_opt_rows else pd.DataFrame(columns=["coreset_size", "num_train_models", "method", *MEAN_COLS, *SE_COLS]),
    METHODS_TO_SHOW,
)


artifact_map = {
    "binary_openllm_helm": (
        binary_table_df,
        "Binary datasets (openllm + helm)",
    ),
    "continuous_cat_main": (
        continuous_table_df,
        "Continuous main datasets",
    ),
    "passk_crossk_source_k1": (
        passk_k1_table_df,
        "pass@k cross-k up to k'=64 (source k=1)",
    ),
    "passk_crossk_source_kopt": (
        passk_opt_table_df,
        f"pass@k cross-k up to k'=64 (source k=opt by {PASSK_OPTIMIZATION_METRIC})",
    ),
}

print("\nSaving outputs:")
for stem, (table_df, caption) in artifact_map.items():
    parquet_path, tex_path = save_table(table_df, stem=stem, caption=caption)
    print(f"- {stem}")
    print(f"  parquet: {parquet_path}")
    print(f"  tex:     {tex_path}")


print("\nTable shapes:")
for stem, (table_df, _) in artifact_map.items():
    print(f"- {stem}: {table_df.shape}")


binary_table_df.head(10), continuous_table_df.head(10), passk_k1_table_df.head(10), passk_opt_table_df.head(10)

Discovered 27 experiment settings across splits: ['binned_interpolation', 'stratified']
Settings per split:
- binned_interpolation: 9
- stratified: 18

Saving outputs:
- binary_openllm_helm
  parquet: /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/binary_openllm_helm.parquet
  tex:     /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/binary_openllm_helm.tex
- continuous_cat_main
  parquet: /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/continuous_cat_main.parquet
  tex:     /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/continuous_cat_main.tex
- passk_crossk_source_k1
  parquet: /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/passk_crossk_source_k1.parquet
  tex:     /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/passk_crossk_source_k1.tex
- passk_crossk_source_kopt
  parquet: /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/passk_crossk_source_kopt.parquet
  tex:     /home/dg22309/Documents/mrmr_project/mrmr/viz/tables/passk_crossk_source_kopt.tex


(  coreset_size num_train_models                     method      RMSE  \
 0           5%               15            random_sampling  0.056968   
 1           5%               15  random_sampling_and_learn  0.049973   
 2           5%               15     sample_first_and_learn  0.062656   
 3           5%               15    random_search_and_learn  0.048967   
 4           5%               15     small_search_and_learn  0.048166   
 5           5%               15                       aipw  0.048683   
 6           5%               15            double_optimize  0.054149   
 7           5%               15                      lasso  0.076305   
 8           5%               15                        pca  0.100531   
 9           5%               15                  metabench  0.163487   
 
         MAE  Kendall_tau  Spearman_rho  pearson_r  execution_time   RMSE_se  \
 0  0.045755     0.741864      0.865799   0.893334        0.009117  0.006478   
 1  0.040116     0.729973      0.87